# Amaliyot 13. Ierarxik klasterlash va DBSCAN

13-interaktiv laboratoriyaga («Ierarxik va DBSCAN») hamroh. Kurs mavzusi: aglomerativ va zichlikka asoslangan klasterlash

Daftar oʻz-oʻzidan yetarli: birinchi katak `common.py` faylini oʻquv korpusi va yordamchi funksiyalar bilan yaratadi. Kataklarni yuqoridan pastga ishga tushiring. Korpus oʻquv (shablon) boʻlib, haqiqiy masalada oʻz CSV faylingizni quyidagi ustunlar bilan qoʻying: `text` va `label`.

In [ ]:
%%writefile common.py
"""Matnlarni tasniflash va klasterlash amaliyotlari uchun umumiy funksiyalar."""
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

TOPICS = {
    "sport": ("oʻyin jamoa gol oʻyinchi murabbiy chempionat turnir gʻalaba magʻlubiyat mavsum stadion muxlis hakam hisob "
              "himoyachi hujumchi darvozabon kubok liga final transfer klub zarba penalti musobaqa sportchi "
              "rekord medal olimpiada mashq durang").split(),
    "siyosat": ("qonun deputat saylov partiya ovoz prezident vazir hukumat parlament islohot byudjet "
                "sanksiya nomzod koalitsiya vazirlik farmon tuzatish loyiha muxolifat referendum diplomatiya "
                "hokim konstitutsiya majlis komissiya muzokara sammit saylovchi mandat vakolat").split(),
    "oshxona": ("shoʻrva retsept tuz goʻsht pishirmoq duxovka sabzavot tova qozon xamir shakar ziravor qaynatmoq qovurmoq "
                "dimlamoq sous bulyon salat oshpaz taom masalliq yogʻ un tuxum sut sarimsoq piyoz garmdori "
                "kartoshka shirinlik pishiriq marinad").split(),
    "texnologiya": ("protsessor dastur ilova smartfon neyrotarmoq algoritm maʼlumot server yangilanish dasturchi "
                    "kompyuter xotira ekran kamera brauzer tarmoq internet bulut kod platforma foydalanuvchi qurilma "
                    "chip batareya noutbuk dasturiy videokarta tizim interfeys shifrlash").split(),
}
NEUTRAL = ("bugun juda yangi aytishadi keyin ham mumkin edi boʻldi maʼlum qilindi kecha yaqinda hali hammasi "
           "bir nechta muhim birinchi").split()


def make_corpus(n_per_class=150, noise=0.3, length=(6, 11), seed=0):
    """4 mavzudagi qisqa «hujjatlar»dan iborat oʻquv korpusi.
    noise: oʻz mavzusidan boʻlmagan soʻzlar ulushi (neytral yoki begona). noise qancha katta boʻlsa, masala shuncha qiyin."""
    rng = random.Random(seed)
    names = list(TOPICS)
    rows = []
    for label in names:
        others = [t for t in names if t != label]
        for _ in range(n_per_class):
            words = []
            for _ in range(rng.randint(*length)):
                u = rng.random()
                if u < noise * 0.5:
                    words.append(rng.choice(NEUTRAL))
                elif u < noise:
                    words.append(rng.choice(TOPICS[rng.choice(others)]))
                else:
                    words.append(rng.choice(TOPICS[label]))
            rows.append((" ".join(words), label))
    df = pd.DataFrame(rows, columns=["text", "label"])
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def project_2d(X, seed=0):
    """TF-IDF siyrak matritsasini tekislikka proyeksiyalash (TruncatedSVD)."""
    from sklearn.decomposition import TruncatedSVD
    return TruncatedSVD(2, random_state=seed).fit_transform(X)


def scatter2d(Z, labels, title="", ax=None):
    ax = ax or plt.gca()
    codes, uniq = pd.factorize(pd.Series(labels))
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap="tab10", s=14, alpha=0.8)
    ax.set_title(title)
    handles = [plt.Line2D([], [], marker="o", ls="", color=sc.cmap(sc.norm(i)), label=str(u)) for i, u in enumerate(uniq)]
    ax.legend(handles=handles, fontsize=8)
    return ax


def top_terms(coef, terms, class_names, n=8):
    """Har bir sinf uchun eng katta vaznli soʻzlarni chiqaradi (chiziqli modellar)."""
    terms = np.asarray(terms)
    for i, name in enumerate(class_names):
        idx = np.argsort(coef[i])[::-1][:n]
        print(f"{name:12s}", ", ".join(terms[idx]))


In [ ]:
from common import *

# Oʻquv korpusi: 4 ta mavzu. noise — "begona" soʻzlar ulushi (qancha koʻp boʻlsa, shuncha qiyin).
df = make_corpus(n_per_class=100, noise=0.3, seed=0)
# Oʻquv korpusi oʻrniga oʻzingiznikini qoʻying: df = pd.read_csv("texts.csv")  # ustunlar: text, label
print(df.label.value_counts().to_dict())
df.head()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline
from sklearn.cluster import AgglomerativeClustering, DBSCAN
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import NearestNeighbors

X = TfidfVectorizer().fit_transform(df.text); y = df.label.values
Xr = make_pipeline(TruncatedSVD(10, random_state=0), Normalizer(copy=False)).fit_transform(X)   # zich normallashtirilgan vektorlar (10 oʻlcham)
print(Xr.shape)

## 1. Aglomerativ klasterlash: bogʻlanishlarni solishtirish

In [ ]:
rows = []
for link in ["single", "complete", "average", "ward"]:
    lab = AgglomerativeClustering(4, linkage=link).fit_predict(Xr)
    rows.append((link, round(adjusted_rand_score(y, lab), 3)))
pd.DataFrame(rows, columns=["linkage", "ARI"])

## 2. Dendrogramma (60 ta hujjat tanlanmasi)

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
idx = np.random.default_rng(0).choice(len(y), 60, replace=False)
Lk = linkage(Xr[idx], method="ward")
plt.figure(figsize=(13, 4)); dendrogram(Lk, labels=y[idx], leaf_font_size=8); plt.title("Dendrogramma (Ward)"); plt.show()

## 3. DBSCAN: masofalar grafigi boʻyicha ε ni tanlash

In [ ]:
k = 5
dist, _ = NearestNeighbors(n_neighbors=k).fit(Xr).kneighbors(Xr)
plt.plot(np.sort(dist[:, -1])[::-1]); plt.ylabel(f"masofa, k = {k} ta qoʻshnigacha"); plt.xlabel("hujjatlar kamayish tartibida"); plt.title("k-distance: egilishni izlaymiz"); plt.show()

In [ ]:
rows = []
for eps in [0.3, 0.4, 0.5, 0.55, 0.6]:
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(Xr)
    nc = len(set(lab)) - (1 if -1 in lab else 0); noise = (lab == -1).mean()
    ok = lab != -1
    rows.append((eps, nc, round(noise, 2), round(adjusted_rand_score(y[ok], lab[ok]), 3) if ok.sum() > 1 and nc > 1 else np.nan))
pd.DataFrame(rows, columns=["eps", "klasterlar", "shovqin ulushi", "ARI (shovqinsiz)"])

## Topshiriqlar

1. Tanlang `eps` va `min_samples`, bunda DBSCAN aynan 4 ta klaster topadi.
2. Aglomerativ klasterlashni quyidagi metrika bilan solishtiring: `cosine` va `linkage="average"` (parametr nomi `metric` boʻlib, eski nomi `affinity`).
3. DBSCAN ni TF-IDF ning toʻliq matritsasida (`metric="cosine"`) oʻlchamni kamaytirmasdan ishga tushiring. Nima uchun ε ni tanlash ancha qiyin?